In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [10]:
import os
import glob
import numpy as np
import pandas as pd

In [3]:
data_train = pd.read_csv(
    "/content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/metadata/train_metadata.csv"
)
data_test = pd.read_csv(
    "/content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/metadata/test_metadata.csv"
)
data_val = pd.read_csv(
    "/content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints/metadata/validation_metadata.csv"
)



In [6]:
data_train.head()

,image_name,finding_labels,follow_up,patient_id,patient_age,patient_sex,view_position,image_width,image_height,pixel_spacing_x,pixel_spacing_y,label,binary_label,split
0,00004338_003.png,Edema,3,4338,49,M,AP,2500,2048,0.171,0.171,1,Abnormal,train
1,00001922_002.png,No Finding,2,1922,44,M,PA,2992,2991,0.143,0.143,0,Normal,train
2,00025082_006.png,Effusion|Mass,7,25082,60,F,PA,2722,2837,0.143,0.143,1,Abnormal,train
3,00006960_003.png,No Finding,3,6960,16,M,PA,2048,2500,0.171,0.171,0,Normal,train
4,00012094_058.png,Emphysema,58,12094,63,F,AP,2500,2048,0.168,0.168,1,Abnormal,train


In [5]:
data_train.columns

Index(['image_name', 'finding_labels', 'follow_up', 'patient_id',
       'patient_age', 'patient_sex', 'view_position', 'image_width',
       'image_height', 'pixel_spacing_x', 'pixel_spacing_y', 'label',
       'binary_label', 'split'],
      dtype='object')

In [8]:
EXCLUDED = [
    "finding_labels",
    "patient_id",
    "binary_label",
    "image_width",
    "image_height",
    "pixel_spacing_x",
    "pixel_spacing_y",
    'split'
]
data_train.drop(columns=EXCLUDED, inplace=True)
data_test.drop(columns=EXCLUDED, inplace=True)
data_val.drop(columns=EXCLUDED, inplace=True)

In [9]:
data_train.columns

Index(['image_name', 'follow_up', 'patient_age', 'patient_sex',
       'view_position', 'label'],
      dtype='object')

In [13]:
OUTPUT_ROOT = "/content/drive/MyDrive/Reliable-Chest-X-Ray-AI/embeddings_checkpoints"


def load_embeddings(split):

    split_dir = os.path.join(OUTPUT_ROOT, split)

    npz_files = sorted(
        glob.glob(os.path.join(split_dir, "*.npz"))
    )

    all_names = []
    all_embeddings = []

    for file in npz_files:

        data = np.load(file)

        all_names.append(data["image_names"])
        all_embeddings.append(data["embeddings"])

    image_names = np.concatenate(all_names)
    embeddings = np.concatenate(all_embeddings)

    print(split)
    print("Images:", len(image_names))
    print("Embeddings:", embeddings.shape)

    return image_names, embeddings

In [14]:
def build_full_dataset(split, metadata_df):

    image_names, embeddings = load_embeddings(split)

    # أسماء الـ 1024 feature
    embedding_cols = [
        f"embedding_{i}"
        for i in range(embeddings.shape[1])
    ]

    embeddings_df = pd.DataFrame(
        embeddings,
        columns=embedding_cols
    )


    embeddings_df.insert(
        0,
        "image_name",
        image_names
    )

    # Merge metadata + embeddings
    full_df = metadata_df.merge(
        embeddings_df,
        on="image_name",
        how="inner",
        validate="one_to_one"
    )

    return full_df

In [15]:
train_df = build_full_dataset(
    "train",
    data_train
)

val_df = build_full_dataset(
    "validation",
    data_val
)

test_df = build_full_dataset(
    "test",
    data_test
)

train
Images: 14000
Embeddings: (14000, 1024)
validation
Images: 3000
Embeddings: (3000, 1024)
test
Images: 3000
Embeddings: (3000, 1024)


In [16]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print(
    "Number of embeddings:",
    len([c for c in train_df.columns if c.startswith("embedding_")])
)

Train: (14000, 1030)
Validation: (3000, 1030)
Test: (3000, 1030)
Number of embeddings: 1024


In [25]:
# ============================================================
# Data Integrity Checks
# ============================================================

datasets = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

for name, df in datasets.items():

    missing_values = df.isna().sum().sum()
    duplicate_rows = df.duplicated().sum()

    print(f"{name}")
    print("-" * 40)
    print(f"Missing values : {missing_values}")
    print(f"Duplicate rows : {duplicate_rows}")
    print()

Train
----------------------------------------
Missing values : 0
Duplicate rows : 0

Validation
----------------------------------------
Missing values : 0
Duplicate rows : 0

Test
----------------------------------------
Missing values : 0
Duplicate rows : 0



In [26]:
# ============================================================
# Save Merged Datasets to Google Drive
# ============================================================


SAVE_DIR = "/content/drive/MyDrive/Reliable-Chest-X-Ray-AI/merged_datasets"

os.makedirs(SAVE_DIR, exist_ok=True)

# Save as CSV
train_df.to_csv(
    os.path.join(SAVE_DIR, "train_merged.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(SAVE_DIR, "validation_merged.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(SAVE_DIR, "test_merged.csv"),
    index=False
)

print("✅ All merged datasets saved successfully!")
print(f"📁 Location: {SAVE_DIR}")

✅ All merged datasets saved successfully!
📁 Location: /content/drive/MyDrive/Reliable-Chest-X-Ray-AI/merged_datasets
